# Unsloth QLoRA fine-tuning on a Colab T4

Runtime → Change runtime type → **T4 GPU** (Unsloth needs CUDA; it does not run on TPU).
Tuned for 16 GB VRAM, fp16 (the T4 has no bf16), 4-bit models of 8B or smaller.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
!df -h /content | tail -1

## 1. Install

In [ ]:
%%capture
!pip install unsloth
# Fallback if the above hits version conflicts:
# !pip install --no-deps bitsandbytes accelerate xformers peft trl triton
# !pip install --no-deps unsloth

## 2. Config

In [ ]:
MODEL_NAME  = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"  # 3B-8B 4-bit models fit a T4
MAX_SEQ_LEN = 2048     # drop to 1024 if you hit out-of-memory errors
BATCH_SIZE  = 2
GRAD_ACCUM  = 4
MAX_STEPS   = 60       # raise (or use num_train_epochs) for a real run
OUTPUT_DIR  = "outputs"

## 3. Load model (import unsloth first)

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,  # auto -> fp16 on T4
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 4. Data

In [ ]:
from datasets import load_dataset

dataset = load_dataset("yahma/alpaca-cleaned", split="train[:2000]")

def fmt(ex):
    prompt = ex["instruction"] + (("\n\n" + ex["input"]) if ex["input"] else "")
    msgs = [{"role":"user","content":prompt},{"role":"assistant","content":ex["output"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

dataset = dataset.map(fmt)

## 5. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        warmup_steps=5,
        fp16=True, bf16=False,   # T4: fp16 only
        optim="adamw_8bit",
        logging_steps=5,
        output_dir=OUTPUT_DIR,
        report_to="none",
    ),
)
trainer.train()
print(f"Peak VRAM: {torch.cuda.max_memory_reserved()/1024**3:.1f} GB")

## 6. Inference

In [ ]:
FastLanguageModel.for_inference(model)
msgs = [{"role":"user","content":"Explain LoRA in two sentences."}]
ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=ids, max_new_tokens=128)
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

## 7. Save (Colab disk is wiped when the session ends)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
model.save_pretrained("/content/drive/MyDrive/unsloth_lora")      # LoRA adapters only (small)
tokenizer.save_pretrained("/content/drive/MyDrive/unsloth_lora")

# Optional: GGUF for llama.cpp / Ollama
# model.save_pretrained_gguf("/content/gguf_model", tokenizer, quantization_method="q4_k_m")
# Optional: Hugging Face Hub
# model.push_to_hub("your-username/my-lora", token="hf_...")